# Phase 3 — Affinity Predictor

Trains an ensemble of GINEConv models to predict drug-target binding affinity (pKi).

### Lessons from the previous (failed) approach

| Problem | Why it hurt | Our fix |
|---|---|---|
| GCNConv | Ignores bond features, least expressive GNN | GINEConv (uses edge_attr, provably more powerful) |
| MSE + ranking loss | Ranking doesn't care about values → destroys RMSE | Huber loss (robust MSE, directly optimizes RMSE) |
| Group DRO max-loss | Bounces between groups → val RMSE oscillates wildly | WeightedRandomSampler handles balance (already in Phase 1) |
| Checkpoint on SR-CI | Kaggle scores RMSE, not CI | Checkpoint on best val RMSE |
| No LR schedule | Model can't fine-tune late in training | CosineAnnealingWarmRestarts |
| Ignored fingerprints | Phase 2 computed 2048-bit Morgan FPs, never used | fp_proj branch feeds FP into prediction head |
| 3-model ensemble | Limited diversity | 5 models, different seeds |

## 1. Setup

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.utils.data import WeightedRandomSampler
from lifelines.utils import concordance_index
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
from datetime import date
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [ ]:
# Shared config across Phase 3/4/5
INPUT_DIR    = '/home/nba9/data_competition/processed_data/'
MODEL_DIR    = '/home/nba9/data_competition/saved_models_STAR/' + date.today().strftime("%Y%m%d") + "/"
ENSEMBLE_SIZE = 5
HIDDEN_DIM   = 256
ESM_DIM      = 1280
NODE_FEAT    = 29
EDGE_FEAT    = 7
N_GIN_LAYERS = 4
DROPOUT      = 0.2
BATCH_SIZE   = 128

EPOCHS    = 100
PATIENCE  = 20
LR        = 1e-3

os.makedirs(MODEL_DIR, exist_ok=True)
# save arguments for reproducibility
with open(os.path.join(MODEL_DIR, "config.txt"), "w") as f:
    f.write(f"INPUT_DIR: {INPUT_DIR}\n")
    f.write(f"MODEL_DIR: {MODEL_DIR}\n")
    f.write(f"ENSEMBLE_SIZE: {ENSEMBLE_SIZE}\n")
    f.write(f"HIDDEN_DIM: {HIDDEN_DIM}\n")
    f.write(f"ESM_DIM: {ESM_DIM}\n")
    f.write(f"NODE_FEAT: {NODE_FEAT}\n")
    f.write(f"EDGE_FEAT: {EDGE_FEAT}\n")
    f.write(f"N_GIN_LAYERS: {N_GIN_LAYERS}\n")
    f.write(f"DROPOUT: {DROPOUT}\n")
    f.write(f"BATCH_SIZE: {BATCH_SIZE}\n")
    f.write(f"EPOCHS: {EPOCHS}\n")
    f.write(f"PATIENCE: {PATIENCE}\n")
    f.write(f"LR: {LR}\n")

## 2. Model Architecture

In [10]:
def heteroscedastic_nll_loss(mu, logvar, target):
    """
    Computes the Gaussian NLL loss.
    - precision = exp(-logvar) acts as an attention mechanism, down-weighting 
      the MSE for samples the model determines are highly noisy.
    - The '+ logvar' term acts as a regularization penalty to prevent the model 
      from just predicting infinite uncertainty for everything.
    """
    precision = torch.exp(-logvar)
    loss = 0.5 * precision * (target - mu)**2 + 0.5 * logvar
    return torch.mean(loss)

In [11]:
class DTA_Model(nn.Module):
    """
    Drug-Target Affinity predictor modified for Uncertainty Estimation.
    """

    def __init__(self, node_feat=29, edge_feat=7, esm_dim=1280,
                 hidden_dim=256, n_layers=4, dropout=0.2):
        super().__init__()
        self.dropout = dropout

        # --- Drug graph encoder (GINEConv) ---
        self.edge_proj = nn.Linear(edge_feat, hidden_dim)

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for i in range(n_layers):
            in_dim = node_feat if i == 0 else hidden_dim
            mlp = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINEConv(mlp, edge_dim=hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

        # --- Drug fingerprint encoder ---
        self.fp_proj = nn.Sequential(
            nn.Linear(2048, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # --- Protein encoder ---
        self.prot_proj = nn.Sequential(
            nn.Linear(esm_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # --- Prediction head ---
        # graph(256) + fp(256) + protein(256) = 768
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            # MODIFICATION: Output 2 values (mean, log_variance)
            nn.Linear(hidden_dim // 2, 2),
        )

    def get_compound_embedding(self, data):
        """Extract the compound graph embedding (used by Phase 4 for OOD detection)."""
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        edge_emb = self.edge_proj(edge_attr)
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index, edge_emb)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return global_mean_pool(x, batch)

    def forward(self, data):
        # 1. Drug graph
        drug_graph = self.get_compound_embedding(data)  # (B, hidden)
        # 2. Drug fingerprint
        drug_fp = self.fp_proj(data.fp.squeeze(1))      # (B, hidden)
        # 3. Protein
        prot = self.prot_proj(data.target_emb.squeeze(1))  # (B, hidden)
        # 4. Predict
        combined = torch.cat([drug_graph, drug_fp, prot], dim=-1)
        out = self.head(combined)

        # Split output into mean affinity and aleatoric uncertainty
        mu = out[:, 0]
        logvar = out[:, 1]

        return mu, logvar

## 3. Evaluation

In [15]:
def evaluate(model, loader, device):
    """Returns RMSE, SR-CI, and per-split CIs on the validation set."""
    model.eval()
    all_preds, all_targets = [], []
    all_A, all_B, all_C = [], [], []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            
            # STAR UPDATE: Unpack the dual output. We only evaluate accuracy against the mean (mu).
            mu, _ = model(data)
            
            all_preds.extend(mu.cpu().numpy())
            all_targets.extend(data.y.cpu().numpy())
            
            # Use squeeze() to ensure these are flat 1D arrays 
            all_A.extend(data.in_A.squeeze().cpu().numpy())
            all_B.extend(data.in_B.squeeze().cpu().numpy())
            all_C.extend(data.in_C.squeeze().cpu().numpy())

    p, t = np.array(all_preds), np.array(all_targets)
    rmse = np.sqrt(np.mean((p - t) ** 2))

    def safe_ci(targets, preds, mask):
        mask = np.array(mask) == 1
        if mask.sum() < 2: return 0.5
        return concordance_index(targets[mask], preds[mask])

    ci_a = safe_ci(t, p, all_A)
    ci_b = safe_ci(t, p, all_B)
    ci_c = safe_ci(t, p, all_C)
    
    sr_ci = 0.30 * ci_a + 0.35 * ci_b + 0.35 * ci_c

    return rmse, sr_ci, ci_a, ci_b, ci_c

## 4. Load Data & Train Ensemble

**Why Huber loss?** The Kaggle metric is RMSE, so we want something close to MSE. But real assay data has noisy outliers — a few measurements with huge error can dominate MSE and push the model to overfit noise. Huber loss acts like MSE for small errors (good RMSE) but like MAE for large errors (ignores outliers). Best of both worlds.

In [13]:
print('Loading data...')
train_data = torch.load(os.path.join(INPUT_DIR, 'train_data.pt'))
val_data   = torch.load(os.path.join(INPUT_DIR, 'val_data.pt'))

train_weights = [d.sample_weight.item() for d in train_data]
sampler = WeightedRandomSampler(train_weights, len(train_weights), replacement=True)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {len(train_data)} | Val: {len(val_data)} | Batches/epoch: {len(train_loader)}')
print(f'Training {ENSEMBLE_SIZE}-model ensemble...\n')

Loading data...
Train: 23194 | Val: 2578 | Batches/epoch: 181
Training 5-model ensemble...



In [22]:
all_best_rmse = []
for ens_idx in range(2, ENSEMBLE_SIZE + 1):
    torch.manual_seed(42 + ens_idx * 1000)
    np.random.seed(42 + ens_idx * 1000)

    print(f'=== Model {ens_idx}/{ENSEMBLE_SIZE} ===')

    model = DTA_Model(
        node_feat=NODE_FEAT, edge_feat=EDGE_FEAT, esm_dim=ESM_DIM,
        hidden_dim=HIDDEN_DIM, n_layers=N_GIN_LAYERS, dropout=DROPOUT
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=2, eta_min=1e-6
    )

    best_rmse = float('inf')
    best_sr_ci = 0.0
    best_epoch = 1
    patience_ctr = 0
    
    # --- METRICS TRACKER ---
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, n_samples = 0, 0

        for batch in tqdm(train_loader, desc=f'Ep {epoch:02d}', leave=False):
            batch = batch.to(device)
            optimizer.zero_grad()
            
            mu, logvar = model(batch)
            loss = heteroscedastic_nll_loss(mu, logvar, batch.y)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            total_loss += loss.item() * batch.num_graphs
            n_samples += batch.num_graphs

        scheduler.step()
        avg_loss = total_loss / n_samples

        rmse, sr_ci, ci_a, ci_b, ci_c = evaluate(model, val_loader, device)
        
        # --- LOG METRICS ---
        history.append({
            'Epoch': epoch,
            'Train_Loss': avg_loss,
            'Val_RMSE': rmse,
            'Val_SR_CI': sr_ci,
            'CI_A': ci_a,
            'CI_B': ci_b,
            'CI_C': ci_c
        })

        if epoch % 5 == 0 or epoch == 1:
            print(f'Ep {epoch:03d} | Loss: {avg_loss:.4f} | RMSE: {rmse:.4f} | '
                  f'SR-CI: {sr_ci:.4f} (A:{ci_a:.3f} B:{ci_b:.3f} C:{ci_c:.3f})')

        if rmse < best_rmse:
            best_rmse = rmse
            best_sr_ci = sr_ci
            best_epoch = epoch # Remember the epoch for the plot
            patience_ctr = 0
            torch.save(model.state_dict(), os.path.join(MODEL_DIR, f'model_{ens_idx}.pt'))
        else:
            patience_ctr += 1

        if patience_ctr >= PATIENCE:
            print(f'Early stop ep {epoch}. Best RMSE: {best_rmse:.4f}, SR-CI: {best_sr_ci:.4f}')
            break

    all_best_rmse.append(best_rmse)
    print(f'Model {ens_idx} done: RMSE={best_rmse:.4f}, SR-CI={best_sr_ci:.4f}\n')
    
    # --- SAVE METRICS AND PLOT FOR THIS ENSEMBLE MODEL ---
    df_history = pd.DataFrame(history)
    csv_path = os.path.join(MODEL_DIR, f'training_metrics_model_{ens_idx}.csv')
    df_history.to_csv(csv_path, index=False)
    
    # Generate Dual-Axis Plot
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # Axis 1: Loss & RMSE (Left Y-Axis)
    color1 = 'tab:red'
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss / RMSE', color=color1)
    ax1.plot(df_history['Epoch'], df_history['Train_Loss'], color='lightcoral', linestyle='--', label='Train Loss')
    ax1.plot(df_history['Epoch'], df_history['Val_RMSE'], color='darkred', linewidth=2, label='Val RMSE')
    ax1.tick_params(axis='y', labelcolor=color1)
    
    # Axis 2: SR-CI (Right Y-Axis)
    ax2 = ax1.twinx()  
    color2 = 'tab:blue'
    ax2.set_ylabel('SR-CI Score', color=color2)
    ax2.plot(df_history['Epoch'], df_history['Val_SR_CI'], color='royalblue', linewidth=2, label='Val SR-CI')
    ax2.tick_params(axis='y', labelcolor=color2)
    
    # Highlight the chosen Checkpoint
    ax1.axvline(x=best_epoch, color='forestgreen', linestyle=':', linewidth=2.5, 
                label=f'Saved Checkpoint (Ep {best_epoch})')
    
    plt.title(f'Training Dynamics - Model {ens_idx}\nBest Val RMSE: {best_rmse:.4f} | Best Val SR-CI: {best_sr_ci:.4f}')
    fig.tight_layout()
    
    # Combine legends from both axes
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper center', 
               bbox_to_anchor=(0.5, -0.15), ncol=4, frameon=False)
    
    plot_path = os.path.join(MODEL_DIR, f'training_plot_model_{ens_idx}.png')
    plt.savefig(plot_path, bbox_inches='tight', dpi=300)
    plt.close()
    print(f' -> Saved metrics to {csv_path} and plot to {plot_path}')

print(f'\nEnsemble complete. Individual RMSEs: {[f"{r:.4f}" for r in all_best_rmse]}')
print(f'Mean: {np.mean(all_best_rmse):.4f}')

=== Model 2/5 ===


Ep 001 | Loss: 0.7646 | RMSE: 0.8057 | SR-CI: 0.7647 (A:0.774 B:0.754 C:0.767)


Ep 005 | Loss: 0.2002 | RMSE: 0.7133 | SR-CI: 0.7938 (A:0.801 B:0.788 C:0.793)


Ep 010 | Loss: 0.1227 | RMSE: 0.6952 | SR-CI: 0.7980 (A:0.805 B:0.791 C:0.799)


Ep 015 | Loss: 0.0590 | RMSE: 0.6923 | SR-CI: 0.8079 (A:0.814 B:0.801 C:0.810)


...

Ep 080 | Loss: -0.5552 | RMSE: 0.5839 | SR-CI: 0.8659 (A:0.873 B:0.856 C:0.869)


Ep 085 | Loss: -0.5941 | RMSE: 0.5912 | SR-CI: 0.8613 (A:0.866 B:0.854 C:0.865)


Ep 090 | Loss: -0.6913 | RMSE: 0.5786 | SR-CI: 0.8631 (A:0.869 B:0.854 C:0.868)


Ep 095 | Loss: -0.7214 | RMSE: 0.5751 | SR-CI: 0.8701 (A:0.875 B:0.860 C:0.875)


Ep 100 | Loss: -0.7745 | RMSE: 0.5531 | SR-CI: 0.8683 (A:0.872 B:0.860 C:0.873)
Model 5 done: RMSE=0.5531, SR-CI=0.8683

 -> Saved metrics to /home/nba9/data_competition/saved_models_STAR/training_metrics_model_5.csv and plot to /home/nba9/data_competition/saved_models_STAR/training_plot_model_5.png

Ensemble complete. Individual RMSEs: ['0.5658', '0.5624', '0.5783', '0.6141', '0.5531']
Mean: 0.5747


## 5. Validate Ensemble

Averaging multiple models reduces variance — their individual errors partially cancel out.

In [23]:
# Load all members and compute ensemble val performance
models = []
for i in range(1, ENSEMBLE_SIZE + 1):
    m = DTA_Model(node_feat=NODE_FEAT, edge_feat=EDGE_FEAT, esm_dim=ESM_DIM,
                  hidden_dim=HIDDEN_DIM, n_layers=N_GIN_LAYERS, dropout=DROPOUT).to(device)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'model_{i}.pt'), map_location=device))
    m.eval()
    models.append(m)

per_model_preds = [[] for _ in range(ENSEMBLE_SIZE)]
targets = []

with torch.no_grad():
    for batch in val_loader:
        batch = batch.to(device)
        targets.extend(batch.y.cpu().numpy())
        for i, m in enumerate(models):
            # STAR UPDATE: Only capture the mean prediction (`mu`) for RMSE evaluation
            mu, logvar = m(batch)
            per_model_preds[i].extend(mu.cpu().numpy())

targets = np.array(targets)
ens_pred = np.mean(per_model_preds, axis=0)
ens_rmse = np.sqrt(np.mean((ens_pred - targets) ** 2))

print(f'Ensemble val RMSE: {ens_rmse:.4f}')
print(f'Individual mean:   {np.mean(all_best_rmse):.4f}')
print(f'Ensemble gain:     {np.mean(all_best_rmse) - ens_rmse:.4f}')
print('\nPhase 3 complete → proceed to Phase 4.')

Ensemble val RMSE: 0.5550
Individual mean:   0.5747
Ensemble gain:     0.0198

Phase 3 complete → proceed to Phase 4.


## 6. STAR Analyses

In [ ]:
def predict_with_uncertainty(model, data_loader, device, num_samples=30):
    """
    Runs MC Dropout to compute epistemic and aleatoric uncertainty.
    model.train() is strictly required to keep dropout active.
    """
    model.train() 
    all_mus, all_logvars, all_ids, all_splits = [], [], [], []

    with torch.no_grad():
        for batch in data_loader:
            batch = batch.to(device)
            batch_mus, batch_logvars = [], []
            
            # Stochastic forward passes
            for _ in range(num_samples):
                mu, logvar = model(batch)
                batch_mus.append(mu)
                batch_logvars.append(logvar)

            # Stack dimensions: (num_samples, batch_size)
            batch_mus = torch.stack(batch_mus)       
            batch_logvars = torch.stack(batch_logvars) 

            # Expected predicted affinity
            expected_mu = torch.mean(batch_mus, dim=0)

            # Epistemic Uncertainty (Variance of the predicted means)
            epistemic_var = torch.var(batch_mus, dim=0)

            # Aleatoric Uncertainty (Mean of the predicted variances)
            aleatoric_var = torch.mean(torch.exp(batch_logvars), dim=0)
            
            total_var = epistemic_var + aleatoric_var

            all_mus.extend(expected_mu.cpu().numpy())
            all_logvars.extend(total_var.cpu().numpy()) # Storing total variance
            all_ids.extend(batch.id)       # Assuming id is passed in data
            all_splits.extend(batch.split) # Assuming split (IID, Scaffold, Assay) is passed

    return pd.DataFrame({
        'id': all_ids,
        'split': all_splits,
        'pred_affinity': all_mus,
        'total_variance': all_logvars
    })

# --- POST-MODEL: BUDGET-CONSTRAINED RANKING (A) ---
def select_candidates_for_wetlab(df, budget=100, lambda_risk=1.0, max_uncertainty=2.0):
    """
    Ranks candidates by utility and enforces an abstention policy.
    U(x) = predicted_affinity - (lambda * standard_deviation)
    """
    df['std_dev'] = np.sqrt(df['total_variance'])
    
    # 1. Calculate Expected Utility (Penalize high uncertainty)
    df['utility'] = df['pred_affinity'] - (lambda_risk * df['std_dev'])
    
    # 2. Abstention Policy (Throw out wildly uncertain predictions immediately)
    eligible_df = df[df['total_variance'] <= max_uncertainty]
    rejected_count = len(df) - len(eligible_df)
    print(f"Abstention Policy triggered: Dropped {rejected_count} high-risk candidates.")
    
    # 3. Sort by utility and trim to budget
    ranked_df = eligible_df.sort_values(by='utility', ascending=False)
    return ranked_df.head(budget)

# --- POST-MODEL: TRANSPARENT SHIFT EVALUATION (T & R) ---
def report_degradation(results_df, ground_truth_df):
    """
    Evaluates performance across specific distribution shifts.
    """
    merged = pd.merge(results_df, ground_truth_df, on='id')
    
    print("\n--- Shift-Aware Performance Report (STAR: R) ---")
    splits = merged['split'].unique()
    
    for split in splits:
        subset = merged[merged['split'] == split]
        mse = np.mean((subset['pred_affinity'] - subset['true_affinity'])**2)
        rmse = np.sqrt(mse)
        avg_uncertainty = subset['std_dev'].mean()
        
        print(f"Split: {split.ljust(15)} | RMSE: {rmse:.3f} | Avg Uncertainty (std): {avg_uncertainty:.3f}")
        
    # Highlight degradation
    if 'IID' in splits and 'Scaffold' in splits:
        iid_rmse = np.sqrt(np.mean((merged[merged['split']=='IID']['pred_affinity'] - merged[merged['split']=='IID']['true_affinity'])**2))
        scaff_rmse = np.sqrt(np.mean((merged[merged['split']=='Scaffold']['pred_affinity'] - merged[merged['split']=='Scaffold']['true_affinity'])**2))
        degradation = ((scaff_rmse - iid_rmse) / iid_rmse) * 100
        print(f"\nModel Degradation under Scaffold Shift: {degradation:.1f}%")